# 🛠️ dbt (data build tool) — Course 2: Intermediate dbt

> **Platform:** DataCamp | **Instructor:** Mike Metzger (Data Engineer)
> **Level:** Intermediate | **Stack:** dbt-core · DuckDB · SQL · YAML · Jinja

---

## 📋 About This Course

Building on [Course 1](../Course1_IntroToDbt/dbt_Course1_IntroToDbt.ipynb), this course takes the `nyc_yellow_taxi` project from a working prototype to something that can run in **production**. The first half is about **data quality**: built-in, singular and reusable (generic) tests, and how to trace a failing test back to the bad rows. The second half covers the objects that surround models in a real pipeline — **sources**, **seeds** and **snapshots (SCD Type 2)** — and finishes with **`dbt build`**, the single command that seeds, runs, snapshots and tests the whole project in dependency order.

As in Course 1, each chapter starts with the concepts in my own words, followed by the exercises I solved with the code I wrote and screenshots as evidence.

---

## 📚 Table of Contents

| Chapter | Topic | Key commands / concepts |
|---------|-------|-------------------------|
| [Chapter 1](#ch1) | Testing in dbt | `unique`, `not_null`, `accepted_values`, `relationships`, `dbt test`, singular tests, reusable `{% test %}` |
| [Chapter 2](#ch2) | Implementing dbt in production | `sources:` / `source()`, `dbt seed`, `dbt snapshot` (SCD2), `dbt build`, `dbt docs` |
| [Summary](#summary) | What I can do now | Skills checklist |

> 📚 **Lesson slides:** `Ch1_Testing_in_dbt.pdf` | `Ch2_dbt_in_Production.pdf`
> 🖼️ **Evidence:** every screenshot referenced below lives in [`screenshots/`](./screenshots)

---

<a id="ch1"></a>
## 📌 Chapter 1 — Testing in dbt

> **Dataset:** `taxi_rides_raw` model (NYC Yellow Taxi, Jan 2023)
> **Goal:** Add data-quality validations to the project, find the rows that break them, and fix the model or the data.

---

### 📖 Key concepts

**What a test is.** A dbt test is an assertion about a dbt object (models, sources, seeds) that verifies the data is as expected: no nulls, values in a valid range, valid relationships between tables, or any custom rule. Under the hood, **every test is a query that returns the failing rows** — if it returns zero rows, it passes.

**Three kinds of tests**

| Type | Defined in | Use it when |
|------|-----------|-------------|
| **Built-in** (`unique`, `not_null`, `accepted_values`, `relationships`) | `.yml` file, under `tests:` of a column | Standard checks |
| **Singular** | One `.sql` file in `tests/` | A specific, one-off rule for one model |
| **Generic / reusable** | `{% test %}` block in `tests/generic/*.sql` | The same rule is needed on many models / columns |

**Declaring tests** (any `.yml` under `models/`, e.g. `model_properties.yml` or `schema.yml`):

```yaml
version: 2
models:
  - name: taxi_rides_raw
    columns:
      - name: payment_type
        tests:
          - not_null
          - accepted_values:
              values: [1, 2, 3, 4, 5, 6]
```

**Running and debugging tests**

```bash
dbt test                          # every test in the project
dbt test --select <model_or_test> # only one model / test
```

When a test fails, dbt reports how many rows failed and where its compiled SQL lives (`target/compiled/<project>/…`). I can run that SQL in a database client to inspect the bad rows, then fix the data or the model and re-run `dbt run` / `dbt test`.

**Singular tests** are plain SQL saved in `tests/` that must return the rows that *violate* the rule, and they can use Jinja:

```sql
-- tests/assert_order_total_gte_subtotal.sql
select *
from {{ ref('order') }}
where order_total < subtotal
```

**Reusable (generic) tests** are written once with a Jinja `test` block and applied from the `.yml` like a built-in test. `model` and `column_name` are filled in by dbt, and extra arguments can be added:

```sql
-- tests/generic/check_columns_unequal.sql
{% test check_columns_unequal(model, column_name, column_name2) %}
select * from {{ model }}
where {{ column_name }} = {{ column_name2 }}
{% endtest %}
```

```yaml
columns:
  - name: order_time
    tests:
      - check_columns_unequal:
          column_name2: shipped_time
```

In fact, the four built-in tests are themselves generic tests that ship with dbt.

### 🧪 Exercise 1.1 — Built-in vs non built-in tests

dbt ships exactly four generic tests: `unique`, `not_null`, `accepted_values` and `relationships`. Names such as `less_than`, `is_upper` or `not_in` would have to be written as custom tests (or come from a package).

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_01_builtin_tests.png" alt="dbt built-in tests vs non built-in tests" width="620">

*dbt built-in tests vs non built-in tests*

### 🧪 Exercise 1.2 — Defining tests on a model

**Scenario:** Marketing reported records without a payment type and others with unexpected values. According to the data dictionary, `payment_type` must never be null and must be between 1 and 6, so I declared both rules:

```yaml
# models/taxi_rides/model_properties.yml
models:
  - name: taxi_rides_raw
    columns:
      - name: fare_amount
        tests:
          - not_null
      - name: payment_type
        tests:
          - not_null
          - accepted_values:
              values: [1, 2, 3, 4, 5, 6]
```

```bash
dbt run
dbt test
```

**Result:** 3 data tests executed — **PASS=3 WARN=0 ERROR=0**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_02_tests_on_model.png" alt="Tests declared in model_properties.yml and dbt test passing" width="720">

*Tests declared in model_properties.yml and dbt test passing*

### 🧪 Exercise 1.3 — Finding and fixing bad data

**Scenario:** with the tests in place on the full dataset, `dbt test` failed on `accepted_values` for `payment_type`: the source contains rows with `payment_type = 0`, a value outside the dictionary.

| Step | What I did |
|------|-----------|
| 1 | Ran `dbt test` and scrolled the output to identify the failing validation (`accepted_values` on `payment_type`) |
| 2 | Opened the model that feeds the column |
| 3 | Added a filter so only valid payment types reach the warehouse |
| 4 | Re-ran the model and the tests |

**Model before → after:**

```sql
-- models/taxi_rides/taxi_rides_raw.sql
{{ config(materialized='view') }}

with source_data as (
    select * from read_parquet('yellow_tripdata_2023-01-partial.parquet')
)

select * from source_data
where payment_type in (1, 2, 3, 4, 5, 6)     -- added: drop invalid payment types
```

Filtering at the model level keeps invalid records out of every downstream model, and it is easier to maintain than cleaning the raw source.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_03_bad_data_tests_yml.png" alt="Validations on payment_type" width="300">

*Validations on payment_type*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_04_bad_data_model_before.png" alt="taxi_rides_raw before the fix (no filter)" width="520">

*taxi_rides_raw before the fix (no filter)*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_05_bad_data_model_fixed.png" alt="taxi_rides_raw after the fix (invalid payment types filtered)" width="620">

*taxi_rides_raw after the fix (invalid payment types filtered)*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_06_bad_data_feedback.png" alt="Exercise validated" width="620">

*Exercise validated*

### 🧪 Exercise 1.4 — Steps to build a singular test

The process I follow: determine the validation → write the SQL query → save it as a `.sql` file in `tests/` → `dbt run` → `dbt test`.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_07_singular_test_steps.png" alt="Order of steps to develop a singular test" width="560">

*Order of steps to develop a singular test*

### 🧪 Exercise 1.5 — Singular test: trips with zero duration

**Scenario:** a teammate found trips with a duration of 0. The schema has no duration column, but a zero-length trip means the pickup and drop-off timestamps are identical, so the test returns exactly those rows:

```sql
-- tests/assert_trip_duration_gt_0.sql
select *
from taxi_rides_raw
where tpep_pickup_datetime = tpep_dropoff_datetime
```

```bash
dbt test --select assert_trip_duration_gt_0
```

**Result:** the test failed as intended, surfacing **109 trips with zero duration** that need to be handled upstream.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_08_trip_duration_test.png" alt="Singular test detecting 109 zero-duration trips" width="720">

*Singular test detecting 109 zero-duration trips*

### 🧪 Exercise 1.6 — Equivalent queries for the same test

Since a test must return the **failing** rows, more than one query can implement it. Both `dropoff - pickup = 0` and `pickup = dropoff` return the zero-duration trips (valid tests), while `dropoff > pickup` and `dropoff - pickup >= 0` return the *good* rows and would make the test fail for the wrong reason. On large datasets, the cheapest valid query should be preferred.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_09_valid_test_queries.png" alt="Valid (left) vs invalid (right) queries for the trip-duration test" width="720">

*Valid (left) vs invalid (right) queries for the trip-duration test*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_10_valid_test_queries_feedback.png" alt="Exercise validated" width="400">

*Exercise validated*

### 🧪 Exercise 1.7 — Classifying test types

Generic / reusable: `tests/generic/check_columns_equal.sql`, `accepted_values`, and any file ending with `{% endtest %}`. Singular: a plain query such as `select * from {{ ref('order') }} where fulfilled = 1`, or a file like `tests/assert_orderdate_after_feb_1.sql`. The key insight: built-in tests behave exactly like custom generic tests.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_11_test_types.png" alt="Generic/reusable (left) vs singular (right) tests" width="720">

*Generic/reusable (left) vs singular (right) tests*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_12_test_types_feedback.png" alt="Exercise validated" width="400">

*Exercise validated*

### 🧪 Exercise 1.8 — My first reusable test: `check_gt_0`

**Scenario:** several columns in our datasets must be strictly positive, so I started an in-house test library with a generic test. Remember: it returns the rows that **fail**, i.e. values `<= 0`.

```sql
-- tests/generic/check_gt_0.sql
{% test check_gt_0(model, column_name) %}
select *
from {{ model }}
where {{ column_name }} <= 0
{% endtest %}
```

```yaml
# models/taxi_rides/model_properties.yml
version: 2
models:
  - name: taxi_rides_raw
    columns:
      - name: fare_amount
        tests:
          - check_gt_0
      - name: total_amount
        tests:
          - check_gt_0
```

```bash
dbt run
dbt test
```

**Result:** both tests ran and failed — e.g. **2,948 rows** with non-positive `fare_amount`. The failure is the useful outcome: it tells us the source data or the import model must handle refunds / zero fares before the data is used.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_13_reusable_test_yml.png" alt="check_gt_0 applied to fare_amount and total_amount" width="460">

*check_gt_0 applied to fare_amount and total_amount*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_14_reusable_test_sql.png" alt="Definition of the reusable check_gt_0 test" width="460">

*Definition of the reusable check_gt_0 test*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_15_reusable_test_results.png" alt="dbt test output: 2,948 failing rows in fare_amount" width="720">

*dbt test output: 2,948 failing rows in fare_amount*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_16_reusable_test_feedback.png" alt="Exercise validated" width="460">

*Exercise validated*

### 🧪 Exercise 1.9 — Turning a singular test into a reusable one

**Goal:** generalize `assert_trip_duration_gt_0` so the "two columns must not be equal" rule can be applied to any pair of columns, using an extra argument:

```sql
-- tests/generic/columns_equal.sql
{% test columns_equal(model, column_name, column_name2) %}
select *
from {{ model }}
where {{ column_name }} = {{ column_name2 }}
{% endtest %}
```

```yaml
# models/taxi_rides/model_properties.yml
version: 2
models:
  - name: taxi_rides_raw
    columns:
      - name: tpep_pickup_datetime
        tests:
          - columns_equal:
              column_name2: tpep_dropoff_datetime
```

```bash
dbt run
dbt test
```

**Result:** the generic test returned **109 failing rows — exactly the same as the singular test** in Exercise 1.5, confirming the refactor preserved the logic. This is a workflow I can reuse: prototype a rule as a singular test, then promote it to a generic one when it is needed elsewhere.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_17_columns_equal_test.png" alt="columns_equal generic test reproducing the 109 failing rows" width="720">

*columns_equal generic test reproducing the 109 failing rows*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch1_18_columns_equal_feedback.png" alt="Exercise validated" width="460">

*Exercise validated*

> 💡 **Takeaway:** in dbt, quality rules live next to the models and run with one command. Built-in tests cover the basics, singular tests capture one-off business rules, and generic tests let a team build its own reusable validation library.

---

<a id="ch2"></a>
## 📌 Chapter 2 — Implementing dbt in Production

> **Datasets:** `raw.taxi_rides`, `raw.vehicle_list`, `nynj_zipcodes.csv`
> **Goal:** Declare raw data as sources, load reference data with seeds, track history with snapshots, and run the full pipeline with `dbt build`.

---

### 📖 Key concepts

**Sources.** A source names and describes the raw data loaded by the EL process. Declaring it adds the raw tables to the **lineage graph**, and lets me test and document them. Sources are defined under `sources:` in any `.yml` file and queried with `{{ source('source_name', 'table_name') }}`, which compiles to the real table name:

```yaml
version: 2
sources:
  - name: raw
    tables:
      - name: phone_orders
        columns:
          - name: id
            tests: [not_null, unique]
      - name: web_orders
```

```sql
select * from {{ source('raw', 'phone_orders') }}
union
select * from {{ source('raw', 'web_orders') }}
```

| | Models | Sources |
|---|---|---|
| Declared under | `models:` | `sources:` |
| Referenced with | `{{ ref('model') }}` | `{{ source('src', 'table') }}` |
| Tests, lineage, `.yml` definition | ✅ | ✅ |

**Seeds.** CSV files stored in `seeds/` and loaded into the warehouse with `dbt seed`. They are meant for small, rarely changing reference data (countries, postal codes…), **not** for raw data. They are easy to manage, version-controlled, can be configured (schema, quoting, **column data types**) in `dbt_project.yml` or `seeds/properties.yml`, can be tested, and once loaded behave like a model (`{{ ref('zipcodes') }}`). If no data type is given, dbt infers it.

**Snapshots — SCD Type 2.** A *Slowly Changing Dimension Type 2* (Kimball) keeps every state a record went through, instead of overwriting it. dbt implements it with snapshots, adding `dbt_valid_from` / `dbt_valid_to` to each version (the current version has `dbt_valid_to = null`):

| id | order_status | last_updated | dbt_valid_from | dbt_valid_to |
|----|--------------|--------------|----------------|--------------|
| 1 | Received | 10:45 | 10:45 | 11:15 |
| 1 | Packed | 11:15 | 11:15 | 11:30 |
| 1 | Shipped | 11:30 | 11:30 | *null* |

```sql
-- snapshots/snapshot_orders.sql
{% snapshot snapshot_orders %}
{{ config(
    target_schema='snapshots',
    strategy='timestamp',
    unique_key='id',
    updated_at='last_updated'
) }}
select * from {{ source('raw', 'orders') }}
{% endsnapshot %}
```

`dbt snapshot` captures the changes **since the last run**, so it must be scheduled at least as often as the source changes; downstream models query it with `ref()`.

**`dbt build`** combines the production steps in dependency order: `dbt seed` + `dbt run` + `dbt snapshot` + `dbt test`, testing each node before building what depends on it, so bad data does not propagate. It does **not** run `dbt docs` (nor is it a substitute for `dbt compile`). Useful options: `--select <object>`, `--exclude <object>`, `-d` (debug). For small, incremental changes the individual commands can still be run on their own.

| Layer | dbt object |
|-------|-----------|
| Initial data | sources, seeds |
| Transformations for downstream users | models |
| History | snapshots |
| Quality | tests (built-in, singular, generic) |

### 🧪 Exercise 2.1 — Models vs sources cheat sheet

A cheat sheet for the team: only models use the `models:` section and `ref()`; only sources use the `sources:` key and `source(source_name, table_name)`; both can run tests, appear in data lineage and be defined in `models/model_properties.yml`.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_01_models_vs_sources.png" alt="Properties unique to models, unique to sources and common to both" width="720">

*Properties unique to models, unique to sources and common to both*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_02_models_vs_sources_feedback.png" alt="Exercise validated" width="460">

*Exercise validated*

### 🧪 Exercise 2.2 — Re-implementing the raw data as a dbt source

**Scenario:** instead of reading a hard-coded table, `taxi_rides_raw` should read from a declared source `raw.taxi_rides`, so the raw layer shows up in the lineage.

```yaml
# models/model_properties.yml
version: 2

sources:
  - name: raw
    tables:
      - name: taxi_rides
```

```sql
-- models/taxi_rides/taxi_rides_raw.sql
{{ config(materialized='view') }}

select * from {{ source('raw', 'taxi_rides') }}
```

```bash
dbt run
./datacheck
```

**Result:** the model was rebuilt on top of the source with all columns intact and the expected **300,000 rows**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_03_source_yml.png" alt="Source raw.taxi_rides declared in YAML" width="420">

*Source raw.taxi_rides declared in YAML*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_04_source_model.png" alt="taxi_rides_raw now reads from source('raw', 'taxi_rides')" width="520">

*taxi_rides_raw now reads from source('raw', 'taxi_rides')*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_05_source_datacheck.png" alt="Sample of the materialized data" width="620">

*Sample of the materialized data*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_06_source_row_count.png" alt="Row count validation: 300,000" width="200">

*Row count validation: 300,000*

### 🧪 Exercise 2.3 — What is true about seeds

Seeds are (mostly) CSV files, and their column data types can be defined manually.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_07_seeds_quiz.png" alt="Statements true of dbt seeds" width="460">

*Statements true of dbt seeds*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_08_seeds_feedback.png" alt="Exercise validated" width="520">

*Exercise validated*

### 🧪 Exercise 2.4 — Loading NY/NJ ZIP codes as a seed

**Scenario:** the project needs a lookup of New York & New Jersey ZIP codes (`zipcode,place,state`). ZIP codes such as `07093` have **leading zeros**, which would be lost if dbt inferred the column as an integer.

1. Moved `nynj_zipcodes.csv` into `seeds/`.
2. Forced the column type to text in `seeds/properties.yml`:

```yaml
version: 2

seeds:
  - name: nynj_zipcodes.csv
    config:
      column_types:
        zipcode: varchar(5)
```

3. Loaded and validated it:

```bash
dbt seed
./datacheck
```

**Result:** **2,877 rows × 3 columns** loaded with the ZIP codes preserved as 5-character strings — *"Your data counts are correct!"*.
*Note:* dbt identifies a seed by its file name, so the conventional entry is `name: nynj_zipcodes` (without the extension).

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_09_seed_zipcodes.png" alt="Seed configuration and validated load of 2,877 ZIP codes" width="620">

*Seed configuration and validated load of 2,877 ZIP codes*

### 🧪 Exercise 2.5 — Snapshot implementation process

Create a `.sql` file for the snapshot → add the `{% snapshot %}` directive → choose the `unique_key` → set the `updated_at` column → write the query → run `dbt snapshot` on a regular basis (running it regularly is what actually captures the history).

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_10_snapshot_steps.png" alt="Steps to implement a dbt snapshot" width="560">

*Steps to implement a dbt snapshot*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_11_snapshot_steps_feedback.png" alt="Exercise validated" width="460">

*Exercise validated*

### 🧪 Exercise 2.6 — Diagnosing a snapshot that misses changes

**Case:** the data is updated every hour (≈500k rows added or changed), but the snapshot ran only every 24 h, and the same row can change several times a day. Since a snapshot only sees the state at the moment it runs, all intermediate states were lost.

**Solution:** schedule `dbt snapshot` **hourly, right after the update process completes**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_12_snapshot_issue_question.png" alt="Snapshot troubleshooting scenario" width="720">

*Snapshot troubleshooting scenario*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_13_snapshot_issue_answer.png" alt="Selected solution: run the snapshot hourly after the update" width="720">

*Selected solution: run the snapshot hourly after the update*

### 🧪 Exercise 2.7 — Tracking vehicle changes with a snapshot (SCD2)

**Scenario:** a new dataset maps each taxi license to the vehicle in use (`license_id`, `vehicle_make`, `vehicle_model`, `vehicle_year`, `last_updated`). A license can be reassigned to a different vehicle, so to attribute each historical ride to the right vehicle I need the full history → SCD2 snapshot on the `raw.vehicle_list` source.

```yaml
# models/model_properties.yml
version: 2

sources:
  - name: raw
    tables:
      - name: taxi_rides
      - name: vehicle_list
```

```sql
-- snapshots/vehicle_list_snapshot.sql
{% snapshot vehicle_list_snapshot %}

{{
    config(
      target_schema='main',
      unique_key='license_id',
      strategy='timestamp',
      updated_at='last_updated'
    )
}}

select * from {{ source('raw', 'vehicle_list') }}
{% endsnapshot %}
```

```bash
dbt snapshot
```

**Result:** `1 of 1 OK snapshotted main.vehicle_list_snapshot` — **PASS=1 WARN=0 ERROR=0**.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_14_snapshot_sql.png" alt="vehicle_list_snapshot definition" width="560">

*vehicle_list_snapshot definition*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_15_snapshot_source_yml.png" alt="vehicle_list added to the raw source" width="460">

*vehicle_list added to the raw source*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_16_snapshot_run.png" alt="dbt snapshot completed successfully" width="620">

*dbt snapshot completed successfully*

### 🧪 Exercise 2.8 — What `dbt build` does (and doesn't) run

`dbt build` executes `dbt test`, `dbt seed` and `dbt run` (plus snapshots), but it does **not** run `dbt docs` or `dbt compile`.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_17_dbt_build_scope.png" alt="Sub-commands executed vs not executed by dbt build" width="720">

*Sub-commands executed vs not executed by dbt build*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_18_dbt_build_feedback.png" alt="Exercise validated" width="460">

*Exercise validated*

### 🧪 Exercise 2.9 — Debugging a project that does not compile

**Scenario:** an intern rebuilt the project from scratch and it failed in production. I ran the production command and fixed the errors one by one, checking that names were consistent across the YAML files.

```bash
dbt build
```

**Error 1 — compilation error in the snapshot.** `vehicle_list_snapshot` depended on a source named `new.vehicle_list`, but the source is declared as `raw`:

```sql
-- before
select * from {{ source('new', 'vehicle_list') }}
-- after
select * from {{ source('raw', 'vehicle_list') }}
```

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_19_intern_error1.png" alt="Compilation error: source 'new.vehicle_list' not found" width="720">

*Compilation error: source 'new.vehicle_list' not found*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_20_intern_error1_snapshot.png" alt="Snapshot referencing the wrong source name" width="560">

*Snapshot referencing the wrong source name*

**Error 2 — runtime error in `taxi_rides_raw`.** The model queried a literal table `'taxi_rides'` that does not exist (*Did you mean "raw.taxi_rides"?*), and dbt skipped every child of the model. The fix is to use the declared source:

```sql
-- before
{{ config(materialized='view') }}
select * from 'taxi_rides'

-- after
{{ config(materialized='view') }}
select * from {{ source('raw', 'taxi_rides') }}
```

After both fixes, `dbt build` ran cleanly.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_21_intern_error2.png" alt="Runtime error: table taxi_rides does not exist" width="720">

*Runtime error: table taxi_rides does not exist*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_22_intern_error2_before.png" alt="taxi_rides_raw before the fix" width="560">

*taxi_rides_raw before the fix*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_23_intern_error2_fixed.png" alt="taxi_rides_raw after the fix, using source()" width="640">

*taxi_rides_raw after the fix, using source()*

### 🧪 Exercise 2.10 — Putting it all together

Final production run of the project — seeds, models, snapshots and tests in one command — followed by the documentation, which now includes the full lineage from sources and seeds to models and snapshots:

```bash
dbt build             # seeds + models + snapshots + tests, in DAG order
dbt docs generate     # build the documentation site
dbt docs serve        # browse it on http://localhost:8080 (Ctrl+C to exit)
```

**Result:** the complete `nyc_yellow_taxi` project built successfully and its documentation was served on port 8080.

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_24_build_docs_serve.png" alt="Final project: dbt build and documentation served on port 8080" width="720">

*Final project: dbt build and documentation served on port 8080*

<img src="https://raw.githubusercontent.com/JuanCGJ/dbt/main/Course2_IntermediateDbt/screenshots/ch2_25_final_feedback.png" alt="Course project completed" width="560">

*Course project completed*

> 💡 **Takeaway:** a production-ready dbt project is more than models: sources make raw data visible and testable, seeds version small reference tables, snapshots preserve history, and `dbt build` runs all of it with tests acting as quality gates between layers.

---

<a id="summary"></a>
## ✅ Summary — What I Can Do After This Course

| Skill | Evidence in this notebook |
|-------|---------------------------|
| Declare built-in tests (`not_null`, `accepted_values`, …) and run them with `dbt test` | Exercises 1.1 – 1.2 |
| Trace a failing test to the bad rows and fix the model | Exercise 1.3 |
| Write singular tests for business rules | Exercises 1.4 – 1.6 |
| Build reusable (generic) tests with Jinja, including extra arguments | Exercises 1.7 – 1.9 |
| Declare and query sources with `source()` | Exercises 2.1 – 2.2 |
| Load reference data with seeds and control column types | Exercises 2.3 – 2.4 |
| Implement SCD Type 2 history with snapshots and schedule them correctly | Exercises 2.5 – 2.7 |
| Run a full production pipeline with `dbt build`, debug compilation / runtime errors and publish docs | Exercises 2.8 – 2.10 |

### 🧭 Both courses at a glance

```
raw parquet ──► source: raw.taxi_rides ──► taxi_rides_raw (view, tested) ──► daily metrics models
raw table   ──► source: raw.vehicle_list ──► vehicle_list_snapshot (SCD2)
seeds/      ──► nynj_zipcodes (lookup)
                         └── dbt build  →  dbt docs generate / serve
```

**Previous →** [Course 1: Introduction to dbt](../Course1_IntroToDbt/dbt_Course1_IntroToDbt.ipynb)